In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/ParkingProject/data/melbourne.csv"
df = pd.read_csv(path)

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3309 entries, 0 to 3308
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Lastupdated         3309 non-null   object 
 1   Status_Timestamp    3309 non-null   object 
 2   Zone_Number         3091 non-null   float64
 3   Status_Description  3309 non-null   object 
 4   KerbsideID          3309 non-null   int64  
 5   Location            3309 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 155.2+ KB


In [ ]:
# Convert timestamp
df['timestamp'] = pd.to_datetime(df['Status_Timestamp'])

# Extract lat/lon
df[['lat', 'lon']] = df['Location'].str.split(',', expand=True).astype(float)

# Convert occupancy to binary
df['occupied'] = df['Status_Description'].map({
    'Present': 1,
    'Unoccupied': 0
})

df = df[['timestamp', 'lat', 'lon', 'occupied']]

df.head()

/tmp/ipykernel_1647/4194081218.py:2: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['timestamp'] = pd.to_datetime(df['Status_Timestamp'])


,timestamp,lat,lon,occupied
0,2025-03-25 13:09:20+13:00,-37.816205,144.969789,0
1,2025-03-25 12:56:53+13:00,-37.810200,144.972946,1
2,2025-03-25 13:06:47+13:00,-37.813134,144.970672,0
3,2025-03-25 12:16:22+13:00,-37.816253,144.969811,1
4,2025-03-12 19:42:04+13:00,-37.807998,144.972678,0


In [ ]:
# Convert with utc=True to handle the timezone awareness
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

df['timestamp_hour'] = df['timestamp'].dt.floor('H')

df.head()

/tmp/ipykernel_1647/321165952.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['timestamp_hour'] = df['timestamp'].dt.floor('H')


,timestamp,lat,lon,occupied,timestamp_hour
0,2025-03-25 00:09:20+00:00,-37.816205,144.969789,0,2025-03-25 00:00:00+00:00
1,2025-03-24 23:56:53+00:00,-37.810200,144.972946,1,2025-03-24 23:00:00+00:00
2,2025-03-25 00:06:47+00:00,-37.813134,144.970672,0,2025-03-25 00:00:00+00:00
3,2025-03-24 23:16:22+00:00,-37.816253,144.969811,1,2025-03-24 23:00:00+00:00
4,2025-03-12 06:42:04+00:00,-37.807998,144.972678,0,2025-03-12 06:00:00+00:00


In [ ]:
import requests
import pandas as pd


# 1. CLEAN INPUT DATA
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

# creating proper hourly bucket
df["timestamp_hour"] = df["timestamp"].dt.floor("h")


# 2. FETCH WEATHER DATA
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": -37.8136,
    "longitude": 144.9631,
    "start_date": str(df["timestamp"].dt.date.min()),
    "end_date": str(df["timestamp"].dt.date.max()),
    "hourly": "temperature_2m,precipitation"
}

response = requests.get(url, params=params)
data = response.json()

# safety check to prevent KeyError
if "hourly" not in data:
    raise ValueError(f"Weather API failed: {data}")

weather = pd.DataFrame({
    "timestamp_hour": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"]
})

# ensure clean hourly grid
weather["timestamp_hour"] = weather["timestamp_hour"].dt.floor("h")
weather = weather.drop_duplicates("timestamp_hour").sort_values("timestamp_hour")


# 3. MERGE

merged = df.merge(
    weather,
    on="timestamp_hour",
    how="left",
    validate="m:1"
)


# 4. TIME FEATURES

merged["hour"] = merged["timestamp"].dt.hour
merged["day_of_week"] = merged["timestamp"].dt.dayofweek
merged["is_weekend"] = merged["day_of_week"].isin([5, 6]).astype(int)


# 5. WEATHER FEATURES


# Binary rain
merged["is_rain"] = (merged["precipitation"] > 0).astype(int)

# Rain intensity
merged["rain_level"] = pd.cut(
    merged["precipitation"],
    bins=[-0.01, 0, 2, 100],
    labels=["no_rain", "light_rain", "heavy_rain"]
)

# Handling rare category issue
rain_counts = merged["rain_level"].value_counts()

if "heavy_rain" in rain_counts and rain_counts["heavy_rain"] < 20:
    merged["rain_level"] = pd.cut(
        merged["precipitation"],
        bins=[-0.01, 0, 100],
        labels=["no_rain", "rain"]
    )

# Filling missing weather safely
merged["temperature"] = merged["temperature"].ffill()
merged["precipitation"] = merged["precipitation"].fillna(0)

# Recompute rain features after fill
merged["is_rain"] = (merged["precipitation"] > 0).astype(int)

# Convert categorical to string
merged["rain_level"] = merged["rain_level"].astype(str)

# 6. VALIDATION CHECKS

df_hours = df["timestamp_hour"].nunique()
weather_hours = weather["timestamp_hour"].nunique()
overlap_hours = len(set(df["timestamp_hour"]) & set(weather["timestamp_hour"]))

print("df hours:", df_hours)
print("weather hours:", weather_hours)
print("overlap:", overlap_hours)

print("\nMissing values:")
print(merged.isnull().sum())

# 7. FINAL OUTPUT

merged.head()

df hours: 824
weather hours: 31128
overlap: 824

Missing values:
timestamp         0
lat               0
lon               0
occupied          0
timestamp_hour    0
temperature       0
precipitation     0
hour              0
day_of_week       0
is_weekend        0
is_rain           0
rain_level        0
dtype: int64


,timestamp,lat,lon,occupied,timestamp_hour,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,rain_level
0,2025-03-25 00:09:20+00:00,-37.816205,144.969789,0,2025-03-25 00:00:00+00:00,17.0,0.0,0,1,0,0,no_rain
1,2025-03-24 23:56:53+00:00,-37.810200,144.972946,1,2025-03-24 23:00:00+00:00,15.4,0.0,23,0,0,0,no_rain
2,2025-03-25 00:06:47+00:00,-37.813134,144.970672,0,2025-03-25 00:00:00+00:00,17.0,0.0,0,1,0,0,no_rain
3,2025-03-24 23:16:22+00:00,-37.816253,144.969811,1,2025-03-24 23:00:00+00:00,15.4,0.0,23,0,0,0,no_rain
4,2025-03-12 06:42:04+00:00,-37.807998,144.972678,0,2025-03-12 06:00:00+00:00,26.1,0.2,6,2,0,1,rain


In [ ]:
print("\nRain distribution:")
print(merged["rain_level"].value_counts())

print("\nRain impact:")
print(merged.groupby("rain_level")["occupied"].mean())


Rain distribution:
rain_level
no_rain    3091
rain        218
Name: count, dtype: int64

Rain impact:
rain_level
no_rain    0.593983
rain       0.541284
Name: occupied, dtype: float64


In [ ]:
print(merged["timestamp_hour"].head(10))
print(weather["timestamp_hour"].head(10))

0   2025-03-25 00:00:00+00:00
1   2025-03-24 23:00:00+00:00
2   2025-03-25 00:00:00+00:00
3   2025-03-24 23:00:00+00:00
4   2025-03-12 06:00:00+00:00
5   2025-03-25 00:00:00+00:00
6   2025-01-14 22:00:00+00:00
7   2025-01-21 01:00:00+00:00
8   2025-01-20 10:00:00+00:00
9   2025-01-20 04:00:00+00:00
Name: timestamp_hour, dtype: datetime64[ns, UTC]
0   2022-09-13 00:00:00+00:00
1   2022-09-13 01:00:00+00:00
2   2022-09-13 02:00:00+00:00
3   2022-09-13 03:00:00+00:00
4   2022-09-13 04:00:00+00:00
5   2022-09-13 05:00:00+00:00
6   2022-09-13 06:00:00+00:00
7   2022-09-13 07:00:00+00:00
8   2022-09-13 08:00:00+00:00
9   2022-09-13 09:00:00+00:00
Name: timestamp_hour, dtype: datetime64[ns, UTC]


In [ ]:
import requests
import pandas as pd


# 1. GET PUBLIC HOLIDAYS (AU)

def get_holidays(country_code, years):
    all_holidays = []

    for year in years:
        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
        response = requests.get(url)

        # safety check
        if response.status_code != 200:
            continue

        data = response.json()

        for item in data:
            all_holidays.append(item["date"])

    df_holidays = pd.DataFrame(all_holidays, columns=["date"])

    # ensure datetime
    df_holidays["date"] = pd.to_datetime(df_holidays["date"]).dt.date

    df_holidays["is_public_holiday"] = 1

    return df_holidays



# 2. BUILD HOLIDAY TABLE

holiday_df = get_holidays("AU", [2023, 2024, 2025, 2026])



# 3. PREP MAIN DATASET

merged["timestamp"] = pd.to_datetime(merged["timestamp"], utc=True)
merged["timestamp_hour"] = merged["timestamp"].dt.floor("h")

# create date column properly
merged["date"] = merged["timestamp"].dt.date



# 4. MERGE HOLIDAYS

merged = merged.merge(holiday_df, on="date", how="left")

merged["is_public_holiday"] = merged["is_public_holiday"].fillna(0).astype(int)



# 5. FEATURE ENGINEERING

merged["hour"] = merged["timestamp"].dt.hour
merged["day_of_week"] = merged["timestamp"].dt.dayofweek
merged["is_weekend"] = merged["day_of_week"].isin([5, 6]).astype(int)


# CONVERT TO HOURLY (MATCH GEOLOG)


melbourne_final = merged.groupby(
    ["timestamp_hour", "lat", "lon"]
).agg(
    occupancy_rate=("occupied", "mean"),
    total_records=("occupied", "count"),
    occupied_count=("occupied", "sum"),

    # features
    temperature=("temperature", "first"),
    precipitation=("precipitation", "first"),
    is_rain=("is_rain", "first"),
    rain_level=("rain_level", "first"),
    is_public_holiday=("is_public_holiday", "first"),
    hour=("hour", "first"),
    day_of_week=("day_of_week", "first"),
    is_weekend=("is_weekend", "first")
).reset_index()


# 6. QUICK CHECK

print(merged["is_public_holiday"].value_counts())
print(merged.groupby("is_public_holiday")["occupied"].mean())

merged.head()

is_public_holiday
0    3134
1     287
Name: count, dtype: int64
is_public_holiday
0    0.593172
1    0.529617
Name: occupied, dtype: float64


,timestamp,lat,lon,occupied,timestamp_hour,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,rain_level,date,is_public_holiday
0,2025-03-25 00:09:20+00:00,-37.816205,144.969789,0,2025-03-25 00:00:00+00:00,17.0,0.0,0,1,0,0,no_rain,2025-03-25,0
1,2025-03-24 23:56:53+00:00,-37.810200,144.972946,1,2025-03-24 23:00:00+00:00,15.4,0.0,23,0,0,0,no_rain,2025-03-24,0
2,2025-03-25 00:06:47+00:00,-37.813134,144.970672,0,2025-03-25 00:00:00+00:00,17.0,0.0,0,1,0,0,no_rain,2025-03-25,0
3,2025-03-24 23:16:22+00:00,-37.816253,144.969811,1,2025-03-24 23:00:00+00:00,15.4,0.0,23,0,0,0,no_rain,2025-03-24,0
4,2025-03-12 06:42:04+00:00,-37.807998,144.972678,0,2025-03-12 06:00:00+00:00,26.1,0.2,6,2,0,1,rain,2025-03-12,0


In [ ]:
print("SHAPE:", melbourne_final.shape)

print("\nNULLS:")
print(melbourne_final.isnull().sum())

print("\nDUPLICATES:")
print(melbourne_final.duplicated(
    subset=["timestamp_hour", "lat", "lon"]
).sum())

print("\nOCCUPANCY STATS:")
print(melbourne_final["occupancy_rate"].describe())

print("\nRAIN DISTRIBUTION:")
print(melbourne_final["rain_level"].value_counts())

melbourne_final.head()

SHAPE: (3309, 14)

NULLS:
timestamp_hour       0
lat                  0
lon                  0
occupancy_rate       0
total_records        0
occupied_count       0
temperature          0
precipitation        0
is_rain              0
rain_level           0
is_public_holiday    0
hour                 0
day_of_week          0
is_weekend           0
dtype: int64

DUPLICATES:
0

OCCUPANCY STATS:
count    3309.000000
mean        0.590511
std         0.491814
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: occupancy_rate, dtype: float64

RAIN DISTRIBUTION:
rain_level
no_rain    3091
rain        218
Name: count, dtype: int64


,timestamp_hour,lat,lon,occupancy_rate,total_records,occupied_count,temperature,precipitation,is_rain,rain_level,is_public_holiday,hour,day_of_week,is_weekend
0,2022-09-13 04:00:00+00:00,-37.810928,144.976544,0.0,1,0,12.6,0.1,1,rain,0,4,1,0
1,2022-11-20 03:00:00+00:00,-37.803845,144.949580,0.0,1,0,17.8,0.0,0,no_rain,0,3,6,1
2,2022-11-29 23:00:00+00:00,-37.810538,144.978320,0.0,1,0,15.9,0.1,1,rain,0,23,1,0
3,2022-12-30 12:00:00+00:00,-37.800909,144.953274,0.0,1,0,18.9,0.0,0,no_rain,0,12,4,0
4,2023-01-15 17:00:00+00:00,-37.808516,144.952590,0.0,1,0,15.1,0.0,0,no_rain,0,17,6,1


In [ ]:
print(melbourne_final["total_records"].describe())
print("\nUnique per group check:")
print(melbourne_final.groupby(["timestamp_hour","lat","lon"]).size().head(10))

count    3309.000000
mean        1.033847
std         0.185809
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         4.000000
Name: total_records, dtype: float64

Unique per group check:
timestamp_hour             lat         lon       
2022-09-13 04:00:00+00:00  -37.810928  144.976544    1
2022-11-20 03:00:00+00:00  -37.803845  144.949580    1
2022-11-29 23:00:00+00:00  -37.810538  144.978320    1
2022-12-30 12:00:00+00:00  -37.800909  144.953274    1
2023-01-15 17:00:00+00:00  -37.808516  144.952590    1
2023-01-24 22:00:00+00:00  -37.808385  144.953044    1
2023-02-13 04:00:00+00:00  -37.819485  144.945015    1
2023-02-25 03:00:00+00:00  -37.804158  144.949524    1
2023-03-16 22:00:00+00:00  -37.806123  144.952093    1
2023-04-04 09:00:00+00:00  -37.804765  144.961719    1
dtype: int64


In [ ]:

# FINAL DATA VALIDATION


print("\n===== SHAPE =====")
print("merged shape:", merged.shape)

print("\n===== COLUMNS =====")
print(merged.columns)

print("\n===== NULL CHECK =====")
print(merged.isnull().sum())

print("\n===== TIME COVERAGE =====")
print("min time:", merged["timestamp"].min())
print("max time:", merged["timestamp"].max())

print("\n===== UNIQUE HOURS =====")
print("unique hours:", merged["timestamp_hour"].nunique())

print("\n===== OCCUPANCY DISTRIBUTION =====")
print(merged["occupied"].value_counts(normalize=True))

print("\n===== WEATHER CHECK =====")
print("temp min/max:", merged["temperature"].min(), merged["temperature"].max())
print("rain ratio:", merged["is_rain"].mean())

print("\n===== TIME FEATURE CHECK =====")
print(merged.groupby("hour")["occupied"].mean())

print("\n===== SAMPLE ROWS =====")
print(merged.sample(5))


# CONSISTENCY CHECK


# checking duplicate timestamp-hour-location combos
dup = merged.duplicated(subset=["timestamp_hour", "lat", "lon"]).sum()
print("\nDuplicate (timestamp_hour, lat, lon):", dup)

# check if any faulty coordinates
print("\nLat range:", merged["lat"].min(), merged["lat"].max())
print("Lon range:", merged["lon"].min(), merged["lon"].max())


===== SHAPE =====
merged shape: (3421, 14)

===== COLUMNS =====
Index(['timestamp', 'lat', 'lon', 'occupied', 'timestamp_hour', 'temperature',
       'precipitation', 'hour', 'day_of_week', 'is_weekend', 'is_rain',
       'rain_level', 'date', 'is_public_holiday'],
      dtype='object')

===== NULL CHECK =====
timestamp            0
lat                  0
lon                  0
occupied             0
timestamp_hour       0
temperature          0
precipitation        0
hour                 0
day_of_week          0
is_weekend           0
is_rain              0
rain_level           0
date                 0
is_public_holiday    0
dtype: int64

===== TIME COVERAGE =====
min time: 2022-09-13 04:38:23+00:00
max time: 2026-04-01 02:16:55+00:00

===== UNIQUE HOURS =====
unique hours: 824

===== OCCUPANCY DISTRIBUTION =====
occupied
1    0.58784
0    0.41216
Name: proportion, dtype: float64

===== WEATHER CHECK =====
temp min/max: 2.7 37.5
rain ratio: 0.06401636948260743

===== TIME FEATURE CHE

In [ ]:

# CONVERT TO HOURLY MODEL DATA


hourly_df = merged.groupby(
    ["timestamp_hour", "lat", "lon"]
).agg(
    occupancy_rate=("occupied", "mean"),
    total_records=("occupied", "count"),
    occupied_count=("occupied", "sum"),
    temperature=("temperature", "mean"),
    precipitation=("precipitation", "mean"),
    is_rain=("is_rain", "max"),
    is_public_holiday=("is_public_holiday", "max"),
    hour=("hour", "first"),
    day_of_week=("day_of_week", "first"),
    is_weekend=("is_weekend", "first")
).reset_index()


print("shape:", hourly_df.shape)

print("\nnulls:")
print(hourly_df.isnull().sum())

print("\nduplicate check:")
print(hourly_df.duplicated(["timestamp_hour","lat","lon"]).sum())

print("\noccupancy range:")
print(hourly_df["occupancy_rate"].describe())

hourly_df.head()

shape: (3309, 13)

nulls:
timestamp_hour       0
lat                  0
lon                  0
occupancy_rate       0
total_records        0
occupied_count       0
temperature          0
precipitation        0
is_rain              0
is_public_holiday    0
hour                 0
day_of_week          0
is_weekend           0
dtype: int64

duplicate check:
0

occupancy range:
count    3309.000000
mean        0.590511
std         0.491814
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: occupancy_rate, dtype: float64


,timestamp_hour,lat,lon,occupancy_rate,total_records,occupied_count,temperature,precipitation,is_rain,is_public_holiday,hour,day_of_week,is_weekend
0,2022-09-13 04:00:00+00:00,-37.810928,144.976544,0.0,1,0,12.6,0.1,1,0,4,1,0
1,2022-11-20 03:00:00+00:00,-37.803845,144.949580,0.0,1,0,17.8,0.0,0,0,3,6,1
2,2022-11-29 23:00:00+00:00,-37.810538,144.978320,0.0,1,0,15.9,0.1,1,0,23,1,0
3,2022-12-30 12:00:00+00:00,-37.800909,144.953274,0.0,1,0,18.9,0.0,0,0,12,4,0
4,2023-01-15 17:00:00+00:00,-37.808516,144.952590,0.0,1,0,15.1,0.0,0,0,17,6,1


In [ ]:
merged.groupby('is_rain')['occupied'].mean()
merged.groupby('temperature')['occupied'].mean()
merged.groupby('hour')['occupied'].mean()
merged.groupby('is_rain')['occupied'].mean()
merged['precipitation'].describe()
merged.describe()

,lat,lon,occupied,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,is_public_holiday
count,3421.000000,3421.000000,3421.000000,3421.000000,3421.000000,3421.000000,3421.000000,3421.000000,3421.000000,3421.000000
mean,-37.813013,144.962246,0.587840,20.857527,0.073633,6.820813,2.035077,0.095294,0.064016,0.083894
std,0.006892,0.011189,0.492296,6.054103,0.819681,7.680102,1.536476,0.293663,0.244818,0.277269
min,-37.848238,144.936436,0.000000,2.700000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-37.816537,144.954940,0.000000,16.500000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
50%,-37.812858,144.960447,1.000000,21.500000,0.000000,2.000000,2.000000,0.000000,0.000000,0.000000
75%,-37.809826,144.970405,1.000000,25.100000,0.000000,9.000000,2.000000,0.000000,0.000000,0.000000
max,-37.794906,144.988612,1.000000,37.500000,15.200000,23.000000,6.000000,1.000000,1.000000,1.000000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests


# 1. LOAD DATA

geelong_path = "/content/drive/MyDrive/ParkingProject/data/geelong.csv"
df_geelong = pd.read_csv(geelong_path)

# 2. CLEAN DATA

df_geelong["timestamp"] = pd.to_datetime(df_geelong["time"], utc=True, errors="coerce")

df_geelong[["lat", "lon"]] = df_geelong["location"].str.split(",", expand=True)
df_geelong["lat"] = pd.to_numeric(df_geelong["lat"], errors="coerce")
df_geelong["lon"] = pd.to_numeric(df_geelong["lon"], errors="coerce")

df_geelong["occupied"] = df_geelong["park_flag_c"].fillna(0).astype(int)

df_geelong = df_geelong[["timestamp", "lat", "lon", "occupied"]]


# 3. HOURLY BUCKET

df_geelong["timestamp_hour"] = df_geelong["timestamp"].dt.floor("h")


# 4. WEATHER API

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": df_geelong["lat"].mean(),
    "longitude": df_geelong["lon"].mean(),
    "start_date": str(df_geelong["timestamp"].dt.date.min()),
    "end_date": str(df_geelong["timestamp"].dt.date.max()),
    "hourly": "temperature_2m,precipitation"
}

response = requests.get(url, params=params)
data = response.json()

if "hourly" not in data:
    raise ValueError(f"Weather API failed: {data}")

weather_geelong = pd.DataFrame({
    "timestamp_hour": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"]
})

weather_geelong["timestamp_hour"] = weather_geelong["timestamp_hour"].dt.floor("h")
weather_geelong = weather_geelong.drop_duplicates("timestamp_hour")

# 5. MERGE

merged_geelong = df_geelong.merge(
    weather_geelong,
    on="timestamp_hour",
    how="left",
    validate="m:1"
)


# 6. TIME FEATURES

merged_geelong["hour"] = merged_geelong["timestamp"].dt.hour
merged_geelong["day_of_week"] = merged_geelong["timestamp"].dt.dayofweek
merged_geelong["is_weekend"] = merged_geelong["day_of_week"].isin([5, 6]).astype(int)


# 7. WEATHER FEATURES

merged_geelong["is_rain"] = (merged_geelong["precipitation"] > 0).astype(int)

merged_geelong["temperature"] = merged_geelong["temperature"].ffill()
merged_geelong["precipitation"] = merged_geelong["precipitation"].fillna(0)


# 8. RAIN LEVEL (MATCHING MELBOURNE)


merged_geelong["rain_level"] = pd.cut(
    merged_geelong["precipitation"],
    bins=[-0.01, 0, 2, 100],
    labels=["no_rain", "light_rain", "heavy_rain"]
)

# 9. HANDLE RARE CATEGORY

rain_counts = merged_geelong["rain_level"].value_counts()

if "heavy_rain" in rain_counts and rain_counts["heavy_rain"] < 20:
    merged_geelong["rain_level"] = pd.cut(
        merged_geelong["precipitation"],
        bins=[-0.01, 0, 100],
        labels=["no_rain", "rain"]
    )


# 10. PUBLIC HOLIDAY

def get_holidays(country_code, years):
    all_holidays = []

    for year in years:
        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
        r = requests.get(url)

        if r.status_code != 200:
            continue

        data = r.json()

        for item in data:
            all_holidays.append(item["date"])

    df_h = pd.DataFrame(all_holidays, columns=["date"])
    df_h["date"] = pd.to_datetime(df_h["date"]).dt.date
    df_h["is_public_holiday"] = 1
    return df_h


holiday_geelong = get_holidays("AU", [2023, 2024, 2025, 2026])

merged_geelong["date"] = merged_geelong["timestamp"].dt.date

merged_geelong = merged_geelong.merge(
    holiday_geelong,
    on="date",
    how="left"
)

merged_geelong["is_public_holiday"] = merged_geelong["is_public_holiday"].fillna(0).astype(int)

# 11. FINAL CHECK

print("Shape:", merged_geelong.shape)

print("\nRain distribution:")
print(merged_geelong["rain_level"].value_counts())

print("\nOccupancy by rain level:")
print(merged_geelong.groupby("rain_level")["occupied"].mean())

merged_geelong.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (127882, 14)

Rain distribution:
rain_level
no_rain       107678
light_rain     19668
heavy_rain       536
Name: count, dtype: int64

Occupancy by rain level:
rain_level
no_rain       0.520868
light_rain    0.526439
heavy_rain    0.529851
Name: occupied, dtype: float64


/tmp/ipykernel_1647/2903842989.py:147: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(merged_geelong.groupby("rain_level")["occupied"].mean())


,timestamp,lat,lon,occupied,timestamp_hour,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,rain_level,date,is_public_holiday
0,2021-03-13 04:38:19+00:00,NaN,NaN,1,2021-03-13 04:00:00+00:00,17.0,1.7,4,5,1,1,light_rain,2021-03-13,0
1,2021-03-13 09:27:00+00:00,NaN,NaN,1,2021-03-13 09:00:00+00:00,13.9,0.1,9,5,1,1,light_rain,2021-03-13,0
2,2021-03-12 21:27:04+00:00,NaN,NaN,1,2021-03-12 21:00:00+00:00,22.4,0.0,21,4,0,0,no_rain,2021-03-12,0
3,2021-03-12 16:50:19+00:00,NaN,NaN,1,2021-03-12 16:00:00+00:00,22.1,0.0,16,4,0,0,no_rain,2021-03-12,0
4,2021-03-12 23:39:45+00:00,NaN,NaN,1,2021-03-12 23:00:00+00:00,25.0,0.0,23,4,0,0,no_rain,2021-03-12,0


In [ ]:
#Geelong working

import pandas as pd
import requests


# 1. LOAD DATA

path = "/content/drive/MyDrive/ParkingProject/data/geelong.csv"
geelong_raw = pd.read_csv(path)

print("RAW COLUMNS:", geelong_raw.columns)


# 2. FIX TIME

geelong_raw["timestamp"] = pd.to_datetime(
    geelong_raw["time"],
    errors="coerce",
    utc=True
)

# removing bad timestamps
geelong_raw = geelong_raw.dropna(subset=["timestamp"])


# 3. FIXING LOCATIONs

geelong_raw = geelong_raw.dropna(subset=["location"])

geelong_raw[["lat", "lon"]] = geelong_raw["location"].str.split(",", expand=True)
geelong_raw["lat"] = pd.to_numeric(geelong_raw["lat"], errors="coerce")
geelong_raw["lon"] = pd.to_numeric(geelong_raw["lon"], errors="coerce")

geelong_raw = geelong_raw.dropna(subset=["lat", "lon"])


# 4. BASIC OCCUPANCY

geelong_raw["occupied"] = geelong_raw["park_flag_c"].astype(int)


# 5. CREATE HOURLY TIME

geelong_raw["timestamp_hour"] = geelong_raw["timestamp"].dt.floor("h")
geelong_raw["date"] = geelong_raw["timestamp"].dt.date


# 6. CHECK TIME

print("\nTIME CHECK:")
print("Min:", geelong_raw["timestamp"].min())
print("Max:", geelong_raw["timestamp"].max())
print("Unique hours:", geelong_raw["timestamp_hour"].nunique())


# 7. HOURLY AGGREGATION

geelong_hourly = geelong_raw.groupby(
    ["timestamp_hour", "lat", "lon"]
).agg(
    occupancy_rate=("occupied", "mean"),
    total_records=("occupied", "count"),
    occupied_count=("occupied", "sum")
).reset_index()


# 8. WEATHER DATA

lat_mean = geelong_hourly["lat"].mean()
lon_mean = geelong_hourly["lon"].mean()

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat_mean,
    "longitude": lon_mean,
    "start_date": str(geelong_raw["date"].min()),
    "end_date": str(geelong_raw["date"].max()),
    "hourly": "temperature_2m,precipitation"
}

response = requests.get(url, params=params)
data = response.json()

if "hourly" not in data:
    raise ValueError(data)

weather = pd.DataFrame({
    "timestamp_hour": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"]
})

weather["timestamp_hour"] = weather["timestamp_hour"].dt.floor("h")


# 9. MERGE

geelong_final = geelong_hourly.merge(
    weather,
    on="timestamp_hour",
    how="left"
)


# 10. TIME FEATURES

geelong_final["hour"] = geelong_final["timestamp_hour"].dt.hour
geelong_final["day_of_week"] = geelong_final["timestamp_hour"].dt.dayofweek
geelong_final["is_weekend"] = geelong_final["day_of_week"].isin([5, 6]).astype(int)


# 11. RAIN FEATURES

geelong_final["is_rain"] = (geelong_final["precipitation"] > 0).astype(int)

geelong_final["rain_level"] = pd.cut(
    geelong_final["precipitation"],
    bins=[-0.01, 0, 2, 100],
    labels=["no_rain", "light_rain", "heavy_rain"]
)


# FIXED PUBLIC HOLIDAYS



def get_holidays(country_code, years):
    all_holidays = []

    for year in years:
        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
        response = requests.get(url)

        if response.status_code != 200:
            continue

        data = response.json()

        for item in data:
            all_holidays.append(item["date"])

    df_holidays = pd.DataFrame(all_holidays, columns=["date"])
    df_holidays["date"] = pd.to_datetime(df_holidays["date"]).dt.date
    df_holidays["is_public_holiday"] = 1

    return df_holidays


# MATCH DATA YEARS
holiday_df_geo = get_holidays("AU", [2020, 2021])


# MERGE HOLIDAYS


# create date column
geelong_final["date"] = geelong_final["timestamp_hour"].dt.date

geelong_final = geelong_final.merge(
    holiday_df_geo,
    on="date",
    how="left"
)

geelong_final["is_public_holiday"] = (
    geelong_final["is_public_holiday"]
    .fillna(0)
    .astype(int)
)


# 12. CLEAN MISSING WEATHER

geelong_final["temperature"] = geelong_final["temperature"].ffill()
geelong_final["precipitation"] = geelong_final["precipitation"].fillna(0)


# 13. FINAL CHECKS

print("\nFINAL SHAPE:", geelong_final.shape)
print("\nNULLS:")
print(geelong_final.isnull().sum())

print("\nRAIN DISTRIBUTION:")
print(geelong_final["rain_level"].value_counts())

print("\nOCCUPANCY STATS:")
print(geelong_final["occupancy_rate"].describe())

geelong_final.head()

RAW COLUMNS: Index(['_id', 'devicename', 'device_label', 'time', 'park_flag_c',
       'duration_occupied', 'duration_free', '2h_overstay', 'battery_percent',
       'frame_count', 'frame_type', 'status', 'timestamp', 'location'],
      dtype='object')

TIME CHECK:
Min: 2020-03-06 02:57:09+00:00
Max: 2021-08-12 14:03:32+00:00
Unique hours: 9792

FINAL SHAPE: (39557, 15)

NULLS:
timestamp_hour       0
lat                  0
lon                  0
occupancy_rate       0
total_records        0
occupied_count       0
temperature          0
precipitation        0
hour                 0
day_of_week          0
is_weekend           0
is_rain              0
rain_level           0
date                 0
is_public_holiday    0
dtype: int64

RAIN DISTRIBUTION:
rain_level
no_rain       33448
light_rain     5938
heavy_rain      171
Name: count, dtype: int64

OCCUPANCY STATS:
count    39557.000000
mean         0.508981
std          0.318863
min          0.000000
25%          0.375000
50%          0.5

,timestamp_hour,lat,lon,occupancy_rate,total_records,occupied_count,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,rain_level,date,is_public_holiday
0,2020-03-06 02:00:00+00:00,-38.152682,144.363429,0.500000,2,1,18.2,0.0,2,4,0,0,no_rain,2020-03-06,0
1,2020-03-06 03:00:00+00:00,-38.151505,144.364349,0.666667,3,2,18.5,0.0,3,4,0,0,no_rain,2020-03-06,0
2,2020-03-06 03:00:00+00:00,-38.147229,144.356721,0.666667,3,2,18.5,0.0,3,4,0,0,no_rain,2020-03-06,0
3,2020-03-06 04:00:00+00:00,-38.152682,144.363429,1.000000,1,1,18.5,0.0,4,4,0,0,no_rain,2020-03-06,0
4,2020-03-06 04:00:00+00:00,-38.152520,144.363224,0.000000,2,0,18.5,0.0,4,4,0,0,no_rain,2020-03-06,0


In [ ]:

# PUBLIC HOLIDAY VALIDATION


# 1. Distribution check
print("===== PUBLIC HOLIDAY DISTRIBUTION =====")
print(geelong_final["is_public_holiday"].value_counts())

# 2. Sample holiday rows
print("\n===== SAMPLE HOLIDAY ROWS =====")
holiday_rows = geelong_final[geelong_final["is_public_holiday"] == 1]
print(holiday_rows.head(10))

# 3. Impact on occupancy
print("\n===== HOLIDAY IMPACT =====")
print(geelong_final.groupby("is_public_holiday")["occupancy_rate"].mean())

# 4. Date coverage
print("\n===== DATE RANGE =====")
print("Min:", geelong_final["timestamp_hour"].min())
print("Max:", geelong_final["timestamp_hour"].max())

# 5. Manual holiday check
print("\n===== MANUAL HOLIDAY CHECK (2020-12-25) =====")
check_date = pd.to_datetime("2020-12-25").date()

print(
    geelong_final[
        geelong_final["timestamp_hour"].dt.date == check_date
    ][["timestamp_hour", "is_public_holiday"]].head(10)
)

===== PUBLIC HOLIDAY DISTRIBUTION =====
is_public_holiday
0    36478
1     3079
Name: count, dtype: int64

===== SAMPLE HOLIDAY ROWS =====
               timestamp_hour        lat         lon  occupancy_rate  \
153 2020-03-09 00:00:00+00:00 -38.152520  144.363224             0.5   
154 2020-03-09 00:00:00+00:00 -38.152520  144.363224             0.5   
155 2020-03-09 00:00:00+00:00 -38.152520  144.363224             0.5   
156 2020-03-09 00:00:00+00:00 -38.152520  144.363224             0.5   
157 2020-03-09 00:00:00+00:00 -38.151274  144.358184             0.5   
158 2020-03-09 00:00:00+00:00 -38.151274  144.358184             0.5   
159 2020-03-09 00:00:00+00:00 -38.151274  144.358184             0.5   
160 2020-03-09 00:00:00+00:00 -38.151274  144.358184             0.5   
161 2020-03-09 00:00:00+00:00 -38.151008  144.357927             0.5   
162 2020-03-09 00:00:00+00:00 -38.151008  144.357927             0.5   

     total_records  occupied_count  temperature  precipitation  hour

In [ ]:

# 1. BASIC STRUCTURE

print("===== SHAPE =====")
print(geelong_final.shape)

print("\n===== COLUMNS =====")
print(geelong_final.columns.tolist())



# 2. NULL CHECK

print("\n===== NULL CHECK =====")
print(geelong_final.isnull().sum())



# 3. DUPLICATE CHECK

dup = geelong_final.duplicated(subset=["timestamp_hour", "lat", "lon"]).sum()
print("\n===== DUPLICATES (hour,lat,lon) =====")
print(dup)



# 4. TIME COVERAGE

print("\n===== TIME COVERAGE =====")
print("min:", geelong_final["timestamp_hour"].min())
print("max:", geelong_final["timestamp_hour"].max())
print("unique hours:", geelong_final["timestamp_hour"].nunique())



# 5. OCCUPANCY CHECK

print("\n===== OCCUPANCY STATS =====")
print(geelong_final["occupancy_rate"].describe())

print("\n===== OCCUPANCY DISTRIBUTION =====")
print((geelong_final["occupancy_rate"] > 0.5).mean())



# 6. WEATHER CHECK

print("\n===== WEATHER CHECK =====")
print("temp min/max:", geelong_final["temperature"].min(), geelong_final["temperature"].max())
print("rain ratio:", (geelong_final["precipitation"] > 0).mean())



# 7. RAIN FEATURE CHECK

print("\n===== RAIN DISTRIBUTION =====")
print(geelong_final["rain_level"].value_counts())

print("\n===== RAIN vs OCCUPANCY =====")
print(geelong_final.groupby("rain_level")["occupancy_rate"].mean())



# 8. TIME FEATURE CHECK

print("\n===== HOUR PATTERN =====")
print(geelong_final.groupby("hour")["occupancy_rate"].mean().head(10))

print("\n===== WEEKEND EFFECT =====")
print(geelong_final.groupby("is_weekend")["occupancy_rate"].mean())



# 9. LOCATION CHECK

print("\n===== LOCATION RANGE =====")
print("lat:", geelong_final["lat"].min(), "→", geelong_final["lat"].max())
print("lon:", geelong_final["lon"].min(), "→", geelong_final["lon"].max())



# 10. SAMPLE DATA

print("\n===== SAMPLE ROWS =====")
print(geelong_final.sample(5))

===== SHAPE =====
(39557, 15)

===== COLUMNS =====
['timestamp_hour', 'lat', 'lon', 'occupancy_rate', 'total_records', 'occupied_count', 'temperature', 'precipitation', 'hour', 'day_of_week', 'is_weekend', 'is_rain', 'rain_level', 'date', 'is_public_holiday']

===== NULL CHECK =====
timestamp_hour       0
lat                  0
lon                  0
occupancy_rate       0
total_records        0
occupied_count       0
temperature          0
precipitation        0
hour                 0
day_of_week          0
is_weekend           0
is_rain              0
rain_level           0
date                 0
is_public_holiday    0
dtype: int64

===== DUPLICATES (hour,lat,lon) =====
762

===== TIME COVERAGE =====
min: 2020-03-06 02:00:00+00:00
max: 2021-08-12 14:00:00+00:00
unique hours: 9792

===== OCCUPANCY STATS =====
count    39557.000000
mean         0.508981
std          0.318863
min          0.000000
25%          0.375000
50%          0.500000
75%          0.666667
max          1.000000
Na

/tmp/ipykernel_1647/651844262.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(geelong_final.groupby("rain_level")["occupancy_rate"].mean())


In [ ]:
import pandas as pd
import requests
import holidays


# 1. LOAD DATA

path = "/content/drive/MyDrive/ParkingProject/data/brisbane.csv"

brisbane_raw = pd.read_csv(path, sep=";")

print("RAW COLUMNS:", brisbane_raw.columns)


# 2. FIX TIME

brisbane_raw["date"] = pd.to_datetime(
    brisbane_raw["date"],
    errors="coerce"
)

brisbane_raw["hour"] = brisbane_raw["hour"].astype(int)

brisbane_raw["timestamp_hour"] = (
    brisbane_raw["date"] +
    pd.to_timedelta(brisbane_raw["hour"], unit="h")
)

brisbane_raw["timestamp_hour"] = pd.to_datetime(
    brisbane_raw["timestamp_hour"],
    utc=True
)

brisbane_raw = brisbane_raw.dropna(subset=["timestamp_hour"])


# 3. LOCATION

brisbane_raw["location_id"] = brisbane_raw["mobile_zone"].astype(str)

brisbane_raw["lat"] = brisbane_raw["mobile_zone"] % 1000
brisbane_raw["lon"] = brisbane_raw["mobile_zone"] % 500


# 4. OCCUPANCY FIX

# Normalize occupancy_pred into 0–1 range
max_occ = brisbane_raw["occupancy_pred"].max()

brisbane_raw["occupancy_rate"] = brisbane_raw["occupancy_pred"] / max_occ


# 5. AGGREGATE (Geelong type)

brisbane_final = brisbane_raw.groupby(
    ["timestamp_hour", "location_id", "lat", "lon"]
).agg(
    occupancy_rate=("occupancy_rate", "mean"),
    total_records=("occupancy_rate", "count")
).reset_index()


# 6. WEATHER

lat_bne = -27.4698
lon_bne = 153.0251

start_date = brisbane_final["timestamp_hour"].min().strftime("%Y-%m-%d")
end_date = brisbane_final["timestamp_hour"].max().strftime("%Y-%m-%d")

print("\nWeather range:", start_date, "→", end_date)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat_bne,
    "longitude": lon_bne,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": "temperature_2m,precipitation"
}

response = requests.get(url, params=params)
data = response.json()

if "hourly" not in data:
    raise ValueError(data)

weather = pd.DataFrame({
    "timestamp_hour": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"]
})

weather["timestamp_hour"] = weather["timestamp_hour"].dt.floor("h")


# 7. MERGE

brisbane_final = brisbane_final.merge(
    weather,
    on="timestamp_hour",
    how="left"
)


# 8. TIME FEATURES

brisbane_final["hour"] = brisbane_final["timestamp_hour"].dt.hour
brisbane_final["day_of_week"] = brisbane_final["timestamp_hour"].dt.dayofweek
brisbane_final["is_weekend"] = brisbane_final["day_of_week"].isin([5, 6]).astype(int)


# 9. RAIN FEATURES

brisbane_final["is_rain"] = (brisbane_final["precipitation"] > 0).astype(int)

brisbane_final["rain_level"] = pd.cut(
    brisbane_final["precipitation"],
    bins=[-0.01, 0, 2, 100],
    labels=["no_rain", "light_rain", "heavy_rain"]
)


# 10. PUBLIC HOLIDAY (AUSTRALIA)

au_holidays = holidays.AU()

brisbane_final["date"] = brisbane_final["timestamp_hour"].dt.date

brisbane_final["is_public_holiday"] = brisbane_final["date"].isin(au_holidays).astype(int)


# 11. CLEAN WEATHER

brisbane_final["temperature"] = brisbane_final["temperature"].ffill()
brisbane_final["precipitation"] = brisbane_final["precipitation"].fillna(0)


# 12. FINAL CHECKS

print("\n===== FINAL SHAPE =====")
print(brisbane_final.shape)

print("\n===== NULLS =====")
print(brisbane_final.isnull().sum())

print("\n===== TIME RANGE =====")
print("Min:", brisbane_final["timestamp_hour"].min())
print("Max:", brisbane_final["timestamp_hour"].max())

print("\n===== OCCUPANCY STATS =====")
print(brisbane_final["occupancy_rate"].describe())

print("\n===== RAIN DISTRIBUTION =====")
print(brisbane_final["rain_level"].value_counts())

brisbane_final.head()

RAW COLUMNS: Index(['mobile_zone', 'date', 'hour', 'occupancy_pred'], dtype='object')

Weather range: 2025-01-02 → 2025-12-31

===== FINAL SHAPE =====
(2729201, 15)

===== NULLS =====
timestamp_hour       0
location_id          0
lat                  0
lon                  0
occupancy_rate       0
total_records        0
temperature          0
precipitation        0
hour                 0
day_of_week          0
is_weekend           0
is_rain              0
rain_level           0
date                 0
is_public_holiday    0
dtype: int64

===== TIME RANGE =====
Min: 2025-01-02 07:00:00+00:00
Max: 2025-12-31 23:00:00+00:00

===== OCCUPANCY STATS =====
count    2.729201e+06
mean     4.231461e-01
std      3.115591e-01
min      0.000000e+00
25%      2.000000e-01
50%      4.000000e-01
75%      6.000000e-01
max      1.000000e+00
Name: occupancy_rate, dtype: float64

===== RAIN DISTRIBUTION =====
rain_level
no_rain       2334488
light_rain     333891
heavy_rain      60822
Name: count, dtype: in

,timestamp_hour,location_id,lat,lon,occupancy_rate,total_records,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,rain_level,date,is_public_holiday
0,2025-01-02 07:00:00+00:00,4000103,103,103,0.6,1,24.8,0.0,7,3,0,0,no_rain,2025-01-02,0
1,2025-01-02 07:00:00+00:00,4000104,104,104,0.4,1,24.8,0.0,7,3,0,0,no_rain,2025-01-02,0
2,2025-01-02 07:00:00+00:00,4000105,105,105,0.4,1,24.8,0.0,7,3,0,0,no_rain,2025-01-02,0
3,2025-01-02 07:00:00+00:00,4000111,111,111,0.4,1,24.8,0.0,7,3,0,0,no_rain,2025-01-02,0
4,2025-01-02 07:00:00+00:00,4000113,113,113,0.6,1,24.8,0.0,7,3,0,0,no_rain,2025-01-02,0


In [ ]:
### Auckland small dataset

import pandas as pd
import requests


# 1. LOAD DATA

path = "/content/drive/MyDrive/ParkingProject/data/auckland.csv"
akl_raw = pd.read_csv(path)

print("RAW COLUMNS:", akl_raw.columns)


# 2. TIMESTAMP CLEANING

akl_raw["timestamp"] = pd.to_datetime(
    akl_raw["timestamp_utc"],
    dayfirst=True,
    errors="coerce",
    utc=True
)

print("\nNull timestamps:", akl_raw["timestamp"].isnull().sum())
akl_raw = akl_raw.dropna(subset=["timestamp"])


# 3. CARPARK LAT/LON

carpark_coords = {
    "Civic": (-36.8509, 174.7645),
    "Downtown": (-36.8480, 174.7620),
    "Ronwood": (-36.9680, 174.8760),
    "Toka Puia": (-36.8450, 174.7700),
    "Victoria Street": (-36.8502, 174.7638)
}

akl_raw["location_id"] = akl_raw["carpark"]

akl_raw["lat"] = akl_raw["location_id"].map(lambda x: carpark_coords.get(x, (None, None))[0])
akl_raw["lon"] = akl_raw["location_id"].map(lambda x: carpark_coords.get(x, (None, None))[1])

akl_raw = akl_raw.dropna(subset=["lat", "lon"])


# 4. OCCUPANCY TARGET

akl_raw["occupancy_rate"] = akl_raw["occupancy"]


# 5. TIME FEATURES

akl_raw["timestamp_hour"] = akl_raw["timestamp"].dt.floor("h")
akl_raw["date"] = akl_raw["timestamp"].dt.date

print("\nDATE CHECK:")
print("Min:", akl_raw["date"].min())
print("Max:", akl_raw["date"].max())


# 6. HOURLY AGGREGATION GEELONG TYPE

akl_hourly = akl_raw.groupby(
    ["timestamp_hour", "location_id", "lat", "lon"]
).agg(
    occupancy_rate=("occupancy_rate", "mean"),
    total_records=("occupancy_rate", "count")
).reset_index()


# 7. WEATHER (OPEN-METEO)

lat_akl = -36.8485
lon_akl = 174.7633

start_date = str(akl_raw["date"].min())
end_date = str(akl_raw["date"].max())

print("\nWeather range:", start_date, "→", end_date)

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat_akl,
    "longitude": lon_akl,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": "temperature_2m,precipitation"
}

res = requests.get(url, params=params)
data = res.json()

if "hourly" not in data:
    raise ValueError(data)

weather = pd.DataFrame({
    "timestamp_hour": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"]
})

weather["timestamp_hour"] = weather["timestamp_hour"].dt.floor("h")


# 8. MERGE WEATHER

akl_final = akl_hourly.merge(
    weather,
    on="timestamp_hour",
    how="left"
)


# 9. TIME FEATURES

akl_final["hour"] = akl_final["timestamp_hour"].dt.hour
akl_final["day_of_week"] = akl_final["timestamp_hour"].dt.dayofweek
akl_final["is_weekend"] = akl_final["day_of_week"].isin([5, 6]).astype(int)


# 10. RAIN FEATURES TO MATCH GEELONG/MELBOURNE

akl_final["is_rain"] = (akl_final["precipitation"] > 0).astype(int)

akl_final["rain_level"] = pd.cut(
    akl_final["precipitation"],
    bins=[-0.01, 0, 2, 100],
    labels=["no_rain", "light_rain", "heavy_rain"]
)

# fallback rare-class fix
rain_counts = akl_final["rain_level"].value_counts()

if "heavy_rain" in rain_counts and rain_counts["heavy_rain"] < 20:
    akl_final["rain_level"] = pd.cut(
        akl_final["precipitation"],
        bins=[-0.01, 0, 100],
        labels=["no_rain", "rain"]
    )


# 11. CLEAN WEATHER

akl_final["temperature"] = akl_final["temperature"].ffill()
akl_final["precipitation"] = akl_final["precipitation"].fillna(0)


# 12. PUBLIC HOLIDAY (NEW ZEALAND)

def get_holidays(country, years):
    all_days = []

    for y in years:
        url = f"https://date.nager.at/api/v3/PublicHolidays/{y}/{country}"
        r = requests.get(url)
        if r.status_code != 200:
            continue

        for h in r.json():
            all_days.append(h["date"])

    df = pd.DataFrame(all_days, columns=["date"])
    df["date"] = pd.to_datetime(df["date"]).dt.date
    df["is_public_holiday"] = 1
    return df

holiday_df = get_holidays("NZ", [2025])

akl_final["date"] = akl_final["timestamp_hour"].dt.date

akl_final = akl_final.merge(
    holiday_df,
    on="date",
    how="left"
)

akl_final["is_public_holiday"] = akl_final["is_public_holiday"].fillna(0).astype(int)


# 13. FINAL STRUCTURE CHECK

print("\n===== FINAL SHAPE =====")
print(akl_final.shape)

print("\n===== NULLS =====")
print(akl_final.isnull().sum())

print("\n===== TIME RANGE =====")
print("Min:", akl_final["timestamp_hour"].min())
print("Max:", akl_final["timestamp_hour"].max())

print("\n===== RAIN DISTRIBUTION =====")
print(akl_final["rain_level"].value_counts())

print("\n===== HOLIDAY DISTRIBUTION =====")
print(akl_final["is_public_holiday"].value_counts())

print("\n===== OCCUPANCY STATS =====")
print(akl_final["occupancy_rate"].describe())


# 14. SAMPLE OUTPUT

akl_final.head()

RAW COLUMNS: Index(['timestamp_utc', 'carpark', 'available_spaces', 'total_spaces',
       'occupancy', 'status', 'parking_option', 'source_last_updated'],
      dtype='object')

Null timestamps: 0

DATE CHECK:
Min: 2025-09-20
Max: 2025-10-09

Weather range: 2025-09-20 → 2025-10-09


/tmp/ipykernel_1647/2144562241.py:17: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  akl_raw["timestamp"] = pd.to_datetime(



===== FINAL SHAPE =====
(2090, 15)

===== NULLS =====
timestamp_hour       0
location_id          0
lat                  0
lon                  0
occupancy_rate       0
total_records        0
temperature          0
precipitation        0
hour                 0
day_of_week          0
is_weekend           0
is_rain              0
rain_level           0
date                 0
is_public_holiday    0
dtype: int64

===== TIME RANGE =====
Min: 2025-09-20 13:00:00+00:00
Max: 2025-10-09 07:00:00+00:00

===== RAIN DISTRIBUTION =====
rain_level
no_rain       1355
light_rain     675
heavy_rain      60
Name: count, dtype: int64

===== HOLIDAY DISTRIBUTION =====
is_public_holiday
0    1980
1     110
Name: count, dtype: int64

===== OCCUPANCY STATS =====
count    2090.000000
mean        0.342107
std         0.190006
min         0.059500
25%         0.221200
50%         0.312300
75%         0.469700
max         1.000000
Name: occupancy_rate, dtype: float64


,timestamp_hour,location_id,lat,lon,occupancy_rate,total_records,temperature,precipitation,hour,day_of_week,is_weekend,is_rain,rain_level,date,is_public_holiday
0,2025-09-20 13:00:00+00:00,Civic,-36.8509,174.7645,0.1767,1,12.3,0.0,13,5,1,0,no_rain,2025-09-20,0
1,2025-09-20 13:00:00+00:00,Downtown,-36.8480,174.7620,0.4146,1,12.3,0.0,13,5,1,0,no_rain,2025-09-20,0
2,2025-09-20 13:00:00+00:00,Ronwood,-36.9680,174.8760,0.2257,1,12.3,0.0,13,5,1,0,no_rain,2025-09-20,0
3,2025-09-20 13:00:00+00:00,Toka Puia,-36.8450,174.7700,0.0643,1,12.3,0.0,13,5,1,0,no_rain,2025-09-20,0
4,2025-09-20 13:00:00+00:00,Victoria Street,-36.8502,174.7638,0.2559,1,12.3,0.0,13,5,1,0,no_rain,2025-09-20,0


In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor


# 2. COPY DATA

train_df = geelong_final.copy()


# 3. FEATURE SELECTION

features = [
    "lat", "lon",
    "hour", "day_of_week", "is_weekend",
    "temperature", "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday"
]

target = "occupancy_rate"


# 4. ENCODE CATEGORICAL

le = LabelEncoder()
train_df["rain_level"] = le.fit_transform(train_df["rain_level"])


# 5. TRAIN / TEST SPLIT (GEELONG)

X = train_df[features]
y = train_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 6. MODEL

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


# 7. EVALUATION

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print("\n=== GEELONG TEST PERFORMANCE ===")
print("MAE:", mae)
print("RMSE:", rmse)


# 8. FEATURE IMPORTANCE

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(importance)


=== GEELONG TEST PERFORMANCE ===
MAE: 0.2167879632500265
RMSE: 0.2968013645110677

=== FEATURE IMPORTANCE ===
             feature  importance
2               hour    0.202177
0                lat    0.176784
1                lon    0.152268
5        temperature    0.086498
8         rain_level    0.069884
3        day_of_week    0.066846
9  is_public_holiday    0.063952
6      precipitation    0.063302
7            is_rain    0.059377
4         is_weekend    0.058914


In [ ]:

# 1. COPY DATA

val_df = melbourne_final.copy()


# 2. MATCH FEATURES

features = [
    "lat", "lon",
    "hour", "day_of_week", "is_weekend",
    "temperature", "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday"
]

target = "occupancy_rate"


# 3. FIX RAIN LEVEL

# Map Melbourne to Geelong style
val_df["rain_level"] = val_df["rain_level"].replace({
    "rain": "light_rain"
})


# 4. ENCODE USING SAME ENCODER

val_df["rain_level"] = le.transform(val_df["rain_level"])


# 5. PREPARE DATA

X_val = val_df[features]
y_val = val_df[target]


# 6. PREDICT

preds_val = model.predict(X_val)


# 7. EVALUATE

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_val = mean_absolute_error(y_val, preds_val)
rmse_val = np.sqrt(mean_squared_error(y_val, preds_val))

print("\n=== MELBOURNE VALIDATION ===")
print("MAE:", mae_val)
print("RMSE:", rmse_val)


# 8. QUICK CHECK

print("\nPREDICTION SAMPLE:")
print(pd.DataFrame({
    "actual": y_val.values[:10],
    "pred": preds_val[:10]
}))


=== MELBOURNE VALIDATION ===
MAE: 0.46709540550281503
RMSE: 0.5267601807795252

PREDICTION SAMPLE:
   actual      pred
0     0.0  0.419181
1     0.0  0.387138
2     0.0  0.504813
3     0.0  1.232969
4     0.0  1.001151
5     0.0  0.928955
6     0.0  0.740985
7     0.0  0.515918
8     1.0  0.738546
9     1.0  0.728485


In [ ]:
from sklearn.cluster import KMeans


# 1. CREATE CLUSTERS

kmeans = KMeans(n_clusters=5, random_state=42)

geelong_final["location_cluster"] = kmeans.fit_predict(
    geelong_final[["lat", "lon"]]
)

# apply SAME clusters to Melbourne
melbourne_final["location_cluster"] = kmeans.predict(
    melbourne_final[["lat", "lon"]]
)


# 2. FEATURES with cluster

features = [
    "location_cluster",
    "hour", "day_of_week", "is_weekend",
    "temperature", "precipitation",
    "is_rain", "rain_level",
    "is_public_holiday"
]

target = "occupancy_rate"


# 3. ENCODE RAIN LEVEL

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_df = geelong_final.copy()
val_df = melbourne_final.copy()

# fix Melbourne rain
val_df["rain_level"] = val_df["rain_level"].replace({
    "rain": "light_rain"
})

train_df["rain_level"] = le.fit_transform(train_df["rain_level"])

val_df["rain_level"] = val_df["rain_level"].apply(
    lambda x: x if x in le.classes_ else "light_rain"
)
val_df["rain_level"] = le.transform(val_df["rain_level"])


# 4. TRAIN

from sklearn.ensemble import RandomForestRegressor

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


# 5. EVALUATE

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred_train = np.clip(model.predict(X_train), 0, 1)
pred_val = np.clip(model.predict(X_val), 0, 1)

print("\n=== GEELONG ===")
print("MAE:", mean_absolute_error(y_train, pred_train))

print("\n=== MELBOURNE ===")
print("MAE:", mean_absolute_error(y_val, pred_val))


=== GEELONG ===
MAE: 0.20593220759155348

=== MELBOURNE ===
MAE: 0.48657334771148486


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error


# 1. LOAD / COPY DATA

akl = akl_final.copy()


# 2. ENSURE DATETIME CLEAN

akl["timestamp_hour"] = pd.to_datetime(akl["timestamp_hour"], utc=True)


# 3. LOCATION CLUSTER

# must use SAME kmeans trained on Geelong
akl["location_cluster"] = kmeans.predict(
    akl[["lat", "lon"]]
)


# 4. ENSURE RAIN LEVEL IS CLEAN

# make sure categories match training
akl["rain_level"] = akl["rain_level"].astype(str)

# if unseen labels exist → map safely
akl["rain_level"] = akl["rain_level"].replace({
    "rain": "light_rain"
})

# encode using training encoder (le)
akl["rain_level"] = akl["rain_level"].apply(
    lambda x: x if x in le.classes_ else "light_rain"
)
akl["rain_level"] = le.transform(akl["rain_level"])


# 5. FEATURE LIST

features = [
    "location_cluster",
    "hour",
    "day_of_week",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday"
]

target = "occupancy_rate"


# 6. FINAL CHECK

missing = [c for c in features if c not in akl.columns]

if missing:
    raise ValueError(f"Missing columns in Auckland dataset: {missing}")


# 7. PREPARE DATA

X_akl = akl[features]
y_akl = akl[target]


# 8. PREDICTION

akl_pred = model.predict(X_akl)

# clamp predictions
akl_pred = np.clip(akl_pred, 0, 1)


# 9. EVALUATION

mae_akl = mean_absolute_error(y_akl, akl_pred)
rmse_akl = np.sqrt(mean_squared_error(y_akl, akl_pred))

print("\n=== AUCKLAND RESULTS ===")
print("MAE:", mae_akl)
print("RMSE:", rmse_akl)


# 10. SAMPLE CHECK

results = pd.DataFrame({
    "actual": y_akl.values[:10],
    "predicted": akl_pred[:10]
})

print("\n=== SAMPLE PREDICTIONS ===")
print(results)


# 11. STATS CHECK

print("\n=== PREDICTION STATS ===")
print("Mean:", akl_pred.mean())
print("Std:", akl_pred.std())
print("Min:", akl_pred.min(), "Max:", akl_pred.max())


=== AUCKLAND RESULTS ===
MAE: 0.21400578921461377
RMSE: 0.27035857373780187

=== SAMPLE PREDICTIONS ===
    actual  predicted
0  0.17670   0.333775
1  0.41460   0.333775
2  0.22570   0.333775
3  0.06430   0.333775
4  0.25590   0.333775
5  0.33840   0.528904
6  0.50335   0.528904
7  0.22120   0.528904
8  0.25240   0.528904
9  0.50000   0.528904

=== PREDICTION STATS ===
Mean: 0.5055736074069416
Std: 0.13815001071938113
Min: 0.1442287672276059 Max: 0.9330142594358826


In [ ]:

# 1. REQUIRED FEATURES

features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday"
]

# check columns exist
missing = [c for c in features if c not in brisbane_final.columns]
if missing:
    raise ValueError(f"Missing columns in Brisbane data: {missing}")


# 2. PREPARE INPUT

X_bris = brisbane_final[features].copy()

X_bris = pd.get_dummies(X_bris, columns=["rain_level"], drop_first=True)

# align columns with training
X_bris = X_bris.reindex(columns=X_train.columns, fill_value=0)


# 3. MODEL PREDICTION

brisbane_final["predicted"] = model.predict(X_bris)


# 4. BASIC PREDICTION CHECK

print("\n=== BRISBANE PREDICTION STATS ===")
print("Mean:", brisbane_final["predicted"].mean())
print("Std:", brisbane_final["predicted"].std())
print("Min:", brisbane_final["predicted"].min())
print("Max:", brisbane_final["predicted"].max())

print("\n=== SAMPLE PREDICTIONS ===")
print(brisbane_final[["occupancy_rate", "predicted"]].sample(10))



# 5. PATTERN COMPARISON


# HOUR PATTERN

print("\n=== HOUR PATTERN COMPARISON ===")

geo_hour = geelong_final.groupby("hour")["occupancy_rate"].mean()
bris_hour_real = brisbane_final.groupby("hour")["occupancy_rate"].mean()
bris_hour_pred = brisbane_final.groupby("hour")["predicted"].mean()

hour_compare = pd.DataFrame({
    "Geelong": geo_hour,
    "Brisbane_real": bris_hour_real,
    "Brisbane_pred": bris_hour_pred
})

print(hour_compare)



# WEEKEND EFFECT

print("\n=== WEEKEND EFFECT COMPARISON ===")

geo_weekend = geelong_final.groupby("is_weekend")["occupancy_rate"].mean()
bris_weekend_real = brisbane_final.groupby("is_weekend")["occupancy_rate"].mean()
bris_weekend_pred = brisbane_final.groupby("is_weekend")["predicted"].mean()

weekend_compare = pd.DataFrame({
    "Geelong": geo_weekend,
    "Brisbane_real": bris_weekend_real,
    "Brisbane_pred": bris_weekend_pred
})

print(weekend_compare)



# RAIN EFFECT

print("\n=== RAIN EFFECT COMPARISON ===")

geo_rain = geelong_final.groupby("rain_level")["occupancy_rate"].mean()
bris_rain_real = brisbane_final.groupby("rain_level")["occupancy_rate"].mean()
bris_rain_pred = brisbane_final.groupby("rain_level")["predicted"].mean()

rain_compare = pd.DataFrame({
    "Geelong": geo_rain,
    "Brisbane_real": bris_rain_real,
    "Brisbane_pred": bris_rain_pred
})

print(rain_compare)



# HOLIDAY EFFECT

print("\n=== HOLIDAY EFFECT COMPARISON ===")

geo_holiday = geelong_final.groupby("is_public_holiday")["occupancy_rate"].mean()
bris_holiday_real = brisbane_final.groupby("is_public_holiday")["occupancy_rate"].mean()
bris_holiday_pred = brisbane_final.groupby("is_public_holiday")["predicted"].mean()

holiday_compare = pd.DataFrame({
    "Geelong": geo_holiday,
    "Brisbane_real": bris_holiday_real,
    "Brisbane_pred": bris_holiday_pred
})

print(holiday_compare)


=== BRISBANE PREDICTION STATS ===
Mean: 0.4237355849278692
Std: 0.1624770079838532
Min: 0.04177197580169798
Max: 0.87359315596712

=== SAMPLE PREDICTIONS ===
         occupancy_rate  predicted
1831837             0.2   0.612699
1133479             0.4   0.361600
1715154             0.6   0.413593
280144              0.2   0.292062
1201401             0.2   0.267972
2481696             0.8   0.052219
769925              0.6   0.274607
1652932             1.0   0.321141
1233599             0.2   0.477800
1781338             0.6   0.479394

=== HOUR PATTERN COMPARISON ===
       Geelong  Brisbane_real  Brisbane_pred
hour                                        
0     0.524790       0.235622       0.498975
1     0.544098       0.193388       0.474149
2     0.554566       0.206488       0.468740
3     0.505391       0.235824       0.605728
4     0.529747       0.224621       0.639922
5     0.526468       0.218705       0.511768
6     0.473026       0.274420       0.395203
7     0.505459    

/tmp/ipykernel_1647/698038132.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  geo_rain = geelong_final.groupby("rain_level")["occupancy_rate"].mean()
/tmp/ipykernel_1647/698038132.py:92: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bris_rain_real = brisbane_final.groupby("rain_level")["occupancy_rate"].mean()
/tmp/ipykernel_1647/698038132.py:93: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bris_rain_pred = brisban

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ENCODING FUNCTION

def encode_features(df):
    df = df.copy()

    # categorical encoding
    le_rain = LabelEncoder()
    df["rain_level"] = le_rain.fit_transform(df["rain_level"])

    # ensure numeric
    binary_cols = ["is_rain", "is_weekend", "is_public_holiday"]
    for col in binary_cols:
        df[col] = df[col].astype(int)

    return df



# APPLY TO DATASETS

geo = encode_features(geelong_final)
akl = encode_features(akl_final)

In [ ]:

# FEATURES

features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday",
    "lat",
    "lon"
]

target = "occupancy_rate"


# GEELONG TRAINING

X_geo = geo[features]
y_geo = geo[target]

base_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=12
)

base_model.fit(X_geo, y_geo)


# AUCKLAND BASE PREDICTION

X_akl = akl[features]
y_akl = akl[target]

akl["geelong_pred"] = base_model.predict(X_akl)

In [ ]:

# CALIBRATION TARGET (RESIDUALS)

akl["error"] = akl["occupancy_rate"] - akl["geelong_pred"]

X_calib = X_akl
y_calib = akl["error"]

calib_model = RandomForestRegressor(
    n_estimators=150,
    random_state=42,
    max_depth=10
)

calib_model.fit(X_calib, y_calib)


# FINAL PREDICTION

akl["calib_pred"] = calib_model.predict(X_calib)

akl["final_pred"] = akl["geelong_pred"] + akl["calib_pred"]

# clamp
akl["final_pred"] = akl["final_pred"].clip(0, 1)


# EVALUATION

mae_base = mean_absolute_error(y_akl, akl["geelong_pred"])
rmse_base = np.sqrt(mean_squared_error(y_akl, akl["geelong_pred"]))

mae_final = mean_absolute_error(y_akl, akl["final_pred"])
rmse_final = np.sqrt(mean_squared_error(y_akl, akl["final_pred"]))

print("=== BASE (GEELONG → AUCKLAND) ===")
print("MAE:", mae_base)
print("RMSE:", rmse_base)

print("\n=== CALIBRATED MODEL ===")
print("MAE:", mae_final)
print("RMSE:", rmse_final)

print("\n=== IMPROVEMENT ===")
print("MAE improvement:", mae_base - mae_final)
print("RMSE improvement:", rmse_base - rmse_final)

=== BASE (GEELONG → AUCKLAND) ===
MAE: 0.3346191035825467
RMSE: 0.39719750021551387

=== CALIBRATED MODEL ===
MAE: 0.040222734332317336
RMSE: 0.05657087685491335

=== IMPROVEMENT ===
MAE improvement: 0.2943963692502294
RMSE improvement: 0.3406266233606005


In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


# 2. CLEAN + ENCODE AUCKLAND DATA

akl = akl_final.copy()
akl = akl.sort_values("timestamp_hour").reset_index(drop=True)

# encode categorical column safely
rain_encoder = LabelEncoder()
akl["rain_level"] = rain_encoder.fit_transform(akl["rain_level"].astype(str))

# binary columns
binary_cols = ["is_rain", "is_weekend", "is_public_holiday"]
for c in binary_cols:
    akl[c] = akl[c].astype(int)


# 3. FEATURES

features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday",
    "lat",
    "lon"
]

target = "occupancy_rate"


# 4. TRAIN GEELONG BASE MODEL

base_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=12
)

base_model.fit(geo[features], geo[target])


# 5. TIME-BASED SPLIT AUCKLAND

n = len(akl)

train_end = int(n * 0.7)
val_end   = int(n * 0.85)

akl_train = akl.iloc[:train_end].copy()
akl_val   = akl.iloc[train_end:val_end].copy()
akl_test  = akl.iloc[val_end:].copy()


# 6. GEELONG PREDICTIONS (FIXED)

def predict(df):
    return base_model.predict(df[features])

akl_train["geelong_pred"] = predict(akl_train)
akl_val["geelong_pred"]   = predict(akl_val)
akl_test["geelong_pred"]  = predict(akl_test)


# 7. CALIBRATION MODEL (TRAIN ONLY)

akl_train["error"] = akl_train[target] - akl_train["geelong_pred"]

calib_model = RandomForestRegressor(
    n_estimators=150,
    random_state=42,
    max_depth=10
)

calib_model.fit(akl_train[features], akl_train["error"])


# 8. APPLY CALIBRATION

def calibrate(df):
    correction = calib_model.predict(df[features])
    final = df["geelong_pred"] + correction
    return np.clip(final, 0, 1)

akl_train["final_pred"] = calibrate(akl_train)
akl_val["final_pred"]   = calibrate(akl_val)
akl_test["final_pred"]  = calibrate(akl_test)


# 9. EVALUATION

def evaluate(name, df, col):
    mae = mean_absolute_error(df[target], df[col])
    rmse = np.sqrt(mean_squared_error(df[target], df[col]))
    print(f"{name} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")

print("\n=== BASE MODEL (GEELONG TRANSFER) ===")
evaluate("Train", akl_train, "geelong_pred")
evaluate("Val", akl_val, "geelong_pred")
evaluate("Test", akl_test, "geelong_pred")

print("\n=== CALIBRATED MODEL ===")
evaluate("Train", akl_train, "final_pred")
evaluate("Val", akl_val, "final_pred")
evaluate("Test", akl_test, "final_pred")


=== BASE MODEL (GEELONG TRANSFER) ===
Train -> MAE: 0.3419, RMSE: 0.4048
Val -> MAE: 0.2989, RMSE: 0.3460
Test -> MAE: 0.3362, RMSE: 0.4092

=== CALIBRATED MODEL ===
Train -> MAE: 0.0376, RMSE: 0.0514
Val -> MAE: 0.0779, RMSE: 0.1060
Test -> MAE: 0.0657, RMSE: 0.0940


In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor



# 2. PREPROCESSING

def preprocess(df):
    df = df.copy()

    df["rain_level"] = df["rain_level"].astype("category").cat.codes

    for c in ["is_rain", "is_weekend", "is_public_holiday"]:
        df[c] = df[c].astype(int)

    return df


geo = preprocess(geelong_final)
akl = preprocess(akl_final)

akl = akl.sort_values("timestamp_hour").reset_index(drop=True)



# 3. FEATURE ENGINEERING

def add_features(df):
    df = df.copy()

    df["hour_weekend"] = df["hour"] * df["is_weekend"]
    df["rain_hour"] = df["hour"] * df["is_rain"]
    df["temp_hour"] = df["hour"] * df["temperature"]

    return df


geo = add_features(geo)
akl = add_features(akl)



# 4. ENCODE CARPARK

akl["location_id_encoded"] = akl["location_id"].astype("category").cat.codes



# 5. FEATURE SETS

features = [
    "hour",
    "day_of_week",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday",
    "lat",
    "lon",
    "hour_weekend",
    "rain_hour",
    "temp_hour"
]

carpark_features = [
    "location_id_encoded",
    "hour",
    "day_of_week"
]

target = "occupancy_rate"



# 6. GEELONG BASE MODEL

base_model = RandomForestRegressor(
    n_estimators=250,
    max_depth=12,
    random_state=42
)

base_model.fit(geo[features], geo[target])



# 7. AUCKLAND BASE PREDICTION

akl["base_pred"] = base_model.predict(akl[features])
akl["error"] = akl[target] - akl["base_pred"]



# 8. GLOBAL RESIDUAL MODEL (LIGHTGBM)

global_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

global_model.fit(akl[features], akl["error"])



# 9. CARPARK BIAS MODEL

carpark_bias_model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

carpark_bias_model.fit(
    akl[carpark_features],
    akl["error"]
)



# 10. TEMPORAL SMOOTHING

akl["hourly_mean"] = akl.groupby(
    ["location_id", "hour"]
)["occupancy_rate"].transform("mean")



# 11. TRAIN / VAL / TEST SPLIT

n = len(akl)

train_end = int(n * 0.7)
val_end = int(n * 0.85)

train = akl.iloc[:train_end].copy()
val = akl.iloc[train_end:val_end].copy()
test = akl.iloc[val_end:].copy()



# 12. FINAL PREDICTION FUNCTION

def predict(df):
    base = base_model.predict(df[features])

    global_corr = global_model.predict(df[features])

    carpark_corr = carpark_bias_model.predict(df[carpark_features])

    smoothing = 0.3 * df["hourly_mean"].values

    final = base + global_corr + carpark_corr + smoothing

    return np.clip(final, 0, 1)


train["final_pred"] = predict(train)
val["final_pred"] = predict(val)
test["final_pred"] = predict(test)



# 13. EVALUATION

def evaluate(name, df, col):
    mae = mean_absolute_error(df[target], df[col])
    rmse = np.sqrt(mean_squared_error(df[target], df[col]))
    print(f"{name} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")


print("\n==============================")
print(" BASE MODEL (GEELONG TRANSFER)")
print("==============================")

evaluate("Train", train, "base_pred")
evaluate("Val", val, "base_pred")
evaluate("Test", test, "base_pred")


print("\n==============================")
print(" FINAL FIXED HIERARCHICAL MODEL")
print("==============================")

evaluate("Train", train, "final_pred")
evaluate("Val", val, "final_pred")
evaluate("Test", test, "final_pred")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000556 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 461
[LightGBM] [Info] Number of data points in the train set: 2090, number of used features: 13
[LightGBM] [Info] Start training from score -0.294905
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000153 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 36
[LightGBM] [Info] Number of data points in the train set: 2090, number of used features: 3
[LightGBM] [Info] Start training from score -0.294905

 BASE MODEL (GEELONG TRANSFER)
Train -> MAE: 0.3476, RMSE: 0.4117
Val -> MAE: 0.3023, RMSE: 0.3501
Test -> MAE: 0.3244, RMSE: 0.3966

 FINAL FIXED HIERARCHICAL MODEL
Train -> MAE: 0.1722, RMSE: 0.1997
Val -> MAE: 0.1617, RMSE: 0.1881
Test -> MAE: 0.1845, 

In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np
import requests


# 2. LOAD DATA

file_path = "/content/drive/MyDrive/ParkingProject/data/auckland_detailed.csv"

df_raw = pd.read_csv(file_path)

print("\n===== RAW DATA =====")
print("Shape:", df_raw.shape)
print(df_raw.head())



# 3. CLEAN COLUMN NAMES

df_raw.columns = df_raw.columns.str.strip().str.lower()

df_raw = df_raw.rename(columns={
    "weekday_/_weekend?": "weekday_type",
    "no_of_cars": "cars",
    "no_of_available_spaces": "available_spaces"
})



# 4. TIMESTAMP

df_raw["date_parsed"] = pd.to_datetime(
    df_raw["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

df_raw["time"] = df_raw["time"].astype(str).str.strip()

df_raw["timestamp"] = pd.to_datetime(
    df_raw["date_parsed"].astype(str) + " " + df_raw["time"],
    errors="coerce"
)

df = df_raw.dropna(subset=["timestamp"]).copy()



# 5. TARGET

df["occupancy_rate"] = (df["cars"] / df["total_spaces"]).clip(0, 1)
df = df.dropna(subset=["occupancy_rate"])



# 6. LOCATION

df["location_id"] = (
    df["street"].astype(str) + "_" +
    df["section"].astype(str) + "_" +
    df["side"].astype(str)
)

df["location_encoded"] = df["location_id"].astype("category").cat.codes



# 7. TIME FEATURES

df["timestamp_hour"] = df["timestamp"].dt.floor("h")

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# cyclic
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)



# 8. WEATHER

start = df["timestamp_hour"].min()
end = df["timestamp_hour"].max()

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": -36.8485,
    "longitude": 174.7633,
    "start_date": start.strftime("%Y-%m-%d"),
    "end_date": end.strftime("%Y-%m-%d"),
    "hourly": "temperature_2m,precipitation"
}

res = requests.get(url, params=params)
data = res.json()

if "hourly" not in data:
    raise ValueError("Weather API failed:", data)

weather_df = pd.DataFrame({
    "timestamp_hour": pd.to_datetime(data["hourly"]["time"]),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"]
})

# REMOVE TIMEZONE
weather_df["timestamp_hour"] = (
    weather_df["timestamp_hour"]
    .dt.tz_localize(None)
    .dt.floor("h")
)

# ALSO ensure df is same type
df["timestamp_hour"] = df["timestamp_hour"].dt.floor("h")

# merge safely
df = df.merge(weather_df, on="timestamp_hour", how="left")

# fill
df["temperature"] = df["temperature"].ffill()
df["precipitation"] = df["precipitation"].fillna(0)



# 9. WEATHER FEATURES

df["is_rain"] = (df["precipitation"] > 0).astype(int)

def rain_level(x):
    if x == 0:
        return "no_rain"
    elif x <= 2:
        return "light"
    elif x <= 5:
        return "moderate"
    else:
        return "heavy"

df["rain_level"] = df["precipitation"].apply(rain_level)



# FIXED PUBLIC HOLIDAYS


def get_nz_holidays(years):
    all_dates = []

    for y in years:
        url = f"https://date.nager.at/api/v3/PublicHolidays/{y}/NZ"
        r = requests.get(url)

        if r.status_code != 200:
            print(f"Holiday API failed for {y}")
            continue

        data = r.json()

        for h in data:
            all_dates.append(pd.to_datetime(h["date"]).date())

    return set(all_dates)


# get years properly
years = sorted(df["timestamp"].dt.year.unique())

holiday_dates = get_nz_holidays(years)

# create feature correctly
df["is_public_holiday"] = df["timestamp"].dt.date.isin(holiday_dates).astype(int)

print("\nHoliday distribution FIXED:")
print(df["is_public_holiday"].value_counts())



# 11. INTERACTION FEATURES

df["hour_weekend"] = df["hour"] * df["is_weekend"]
df["rain_hour"] = df["hour"] * df["is_rain"]
df["temp_hour"] = df["hour"] * df["temperature"]



# 12. FINAL DATASET

akl_final_ready = df[[
    "timestamp_hour",
    "location_id",
    "location_encoded",
    "occupancy_rate",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "is_public_holiday",
    "hour_sin",
    "hour_cos",
    "hour_weekend",
    "rain_hour",
    "temp_hour"
]].copy()



# 13. CHECKS

print("\n===== FINAL DATA =====")
print("Shape:", akl_final_ready.shape)
print("Locations:", akl_final_ready["location_id"].nunique())

print("\nOccupancy stats:")
print(akl_final_ready["occupancy_rate"].describe())

print("\nNulls:")
print(akl_final_ready.isnull().sum())

print("\nRain distribution:")
print(akl_final_ready["rain_level"].value_counts())

print("\nHoliday distribution:")
print(akl_final_ready["is_public_holiday"].value_counts())


print("\n FULL DATASET READY (WITH WEATHER + HOLIDAYS)")


===== RAW DATA =====
Shape: (29036, 17)
    daypk      pk        date  year        day      time weekday_/_weekend?  \
0  2015-1  642234  2015-08-18  2015  Weekday 1  08:00:00            weekday   
1  2015-1  642234  2015-08-18  2015  Weekday 1  08:00:00            weekday   
2  2015-1  642234  2015-08-18  2015  Weekday 1  08:00:00            weekday   
3  2015-1  642234  2015-08-18  2015  Weekday 1  08:00:00            weekday   
4  2015-1  642234  2015-08-18  2015  Weekday 1  08:00:00            weekday   

   precinct             street                                section   side  \
0         6  Wellesley St West                 Hobson St to Nelson St  north   
1         6  Wellesley St West                 Halsey St to Nelson St  north   
2         6  Wellesley St West                 Halsey St to Nelson St  south   
3         6  Wellesley St West                 Hobson St to Nelson St  south   
4         6          Hobson St  Wellesley St West to Victoria St West   west   

   

In [ ]:

# 1. IMPORTS

import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor



# 2. COPY DATA

df = akl_final_ready.copy()



# 3. REMOVE LEAKAGE FEATURES

leak_cols = ["capacity_utilization", "availability_ratio"]
df = df.drop(columns=[c for c in leak_cols if c in df.columns])



# 4. ENCODE CATEGORICALS

df["rain_level"] = df["rain_level"].astype("category").cat.codes



# 5. SORT PROPERLY

df = df.sort_values(["location_id", "timestamp_hour"])



# 6.  CREATE FUTURE TARGET

df["target_next"] = df.groupby("location_id")["occupancy_rate"].shift(-1)

# remove last row per location
df = df.dropna(subset=["target_next"])



# 7. CREATE LAG FEATURES

df["lag_1"] = df.groupby("location_id")["occupancy_rate"].shift(1)
df["lag_2"] = df.groupby("location_id")["occupancy_rate"].shift(2)
df["lag_3"] = df.groupby("location_id")["occupancy_rate"].shift(3)

df["rolling_mean_3"] = (
    df.groupby("location_id")["occupancy_rate"]
    .rolling(3)
    .mean()
    .reset_index(0, drop=True)
)

# dropping rows without lag history
df = df.dropna()



# 8. SORT FOR TIME SPLIT

df = df.sort_values("timestamp_hour")



# 9. TRAIN / VAL / TEST SPLIT (TIME-BASED)

train_size = int(len(df) * 0.7)
val_size   = int(len(df) * 0.15)

train_df = df.iloc[:train_size]
val_df   = df.iloc[train_size:train_size + val_size]
test_df  = df.iloc[train_size + val_size:]



# 10. FEATURES

features = [
    "location_encoded",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "hour_sin",
    "hour_cos",
    "hour_weekend",
    "rain_hour",
    "temp_hour",

    # TEMPORAL MEMORY
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_3"
]

target = "target_next"

X_train, y_train = train_df[features], train_df[target]
X_val, y_val     = val_df[features], val_df[target]
X_test, y_test   = test_df[features], test_df[target]



# 11. MODEL

model = LGBMRegressor(
    n_estimators=700,
    learning_rate=0.04,
    max_depth=6,
    num_leaves=45,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.2,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="l2"
)



# 12. EVALUATION

def evaluate(name, X, y):
    pred = model.predict(X)
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    print(f"{name} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")
    return pred

print("\n==============================")
print(" FINAL FORECAST MODEL (t → t+1)")
print("==============================")

train_pred = evaluate("Train", X_train, y_train)
val_pred   = evaluate("Validation", X_val, y_val)
test_pred  = evaluate("Test", X_test, y_test)



# 13. FEATURE IMPORTANCE

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== FEATURE IMPORTANCE ===")
print(importance)



# 14. SAMPLE PREDICTIONS

sample = pd.DataFrame({
    "actual_next": y_test.values[:10],
    "predicted_next": test_pred[:10]
})

print("\n=== SAMPLE PREDICTIONS (NEXT HOUR) ===")
print(sample)



# 15. PSEUDO ACCURACY

mae_test = mean_absolute_error(y_test, test_pred)
pseudo_acc = (1 - mae_test) * 100

print("\n=== PSEUDO ACCURACY ===")
print(f"{pseudo_acc:.2f}%")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002251 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1451
[LightGBM] [Info] Number of data points in the train set: 16984, number of used features: 18
[LightGBM] [Info] Start training from score 0.783162
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [ ]:

# 1. COPY OLD DATA

old_df = akl_final.copy()



# 2. ENSURE TIMESTAMP + BASIC TIME FEATURES

old_df = old_df.sort_values(["location_id", "timestamp_hour"])

# recreate time features if missing
if "month" not in old_df.columns:
    old_df["month"] = old_df["timestamp_hour"].dt.month

if "hour_sin" not in old_df.columns:
    old_df["hour_sin"] = np.sin(2 * np.pi * old_df["hour"] / 24)

if "hour_cos" not in old_df.columns:
    old_df["hour_cos"] = np.cos(2 * np.pi * old_df["hour"] / 24)

if "hour_weekend" not in old_df.columns:
    old_df["hour_weekend"] = old_df["hour"] * old_df["is_weekend"]

if "rain_hour" not in old_df.columns:
    old_df["rain_hour"] = old_df["hour"] * old_df["is_rain"]

if "temp_hour" not in old_df.columns:
    old_df["temp_hour"] = old_df["hour"] * old_df["temperature"]



# 3. TARGET (t → t+1)

old_df["target_next"] = old_df.groupby("location_id")["occupancy_rate"].shift(-1)



# 4. LAG FEATURES

old_df["lag_1"] = old_df.groupby("location_id")["occupancy_rate"].shift(1)
old_df["lag_2"] = old_df.groupby("location_id")["occupancy_rate"].shift(2)
old_df["lag_3"] = old_df.groupby("location_id")["occupancy_rate"].shift(3)

old_df["rolling_mean_3"] = (
    old_df.groupby("location_id")["occupancy_rate"]
    .rolling(3)
    .mean()
    .reset_index(0, drop=True)
)



# 5. CLEAN NA (lags + target)

old_df = old_df.dropna()



# 6. LOCATION ENCODING

location_map = dict(zip(
    akl_final_ready["location_id"],
    akl_final_ready["location_encoded"]
))

old_df["location_encoded"] = old_df["location_id"].map(location_map)
old_df["location_encoded"] = old_df["location_encoded"].fillna(-1)



# 7. RAIN LEVEL ENCODING

old_df["rain_level"] = old_df["rain_level"].astype("category").cat.codes



# 8. FINAL FEATURE LIST

features = [
    "location_encoded",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "temperature",
    "precipitation",
    "is_rain",
    "rain_level",
    "hour_sin",
    "hour_cos",
    "hour_weekend",
    "rain_hour",
    "temp_hour",
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_3"
]

# ensuring all features exist
missing = [f for f in features if f not in old_df.columns]
if missing:
    raise ValueError(f"Still missing columns: {missing}")



# 9. PREDICTION

X_old = old_df[features]
y_old = old_df["target_next"]

from sklearn.metrics import mean_absolute_error, mean_squared_error

pred_old = model.predict(X_old)

mae = mean_absolute_error(y_old, pred_old)
rmse = np.sqrt(mean_squared_error(y_old, pred_old))

print("\n==============================")
print(" EXTERNAL VALIDATION (OLD AKL)")
print("==============================")

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

pseudo_acc = (1 - mae) * 100
print(f"Pseudo Accuracy: {pseudo_acc:.2f}%")



# 10. SAMPLE OUTPUT

sample = pd.DataFrame({
    "actual_next": y_old.values[:10],
    "predicted_next": pred_old[:10]
})

print("\n=== SAMPLE PREDICTIONS ===")
print(sample)


 EXTERNAL VALIDATION (OLD AKL)
MAE: 0.1005
RMSE: 0.1332
Pseudo Accuracy: 89.95%

=== SAMPLE PREDICTIONS ===
   actual_next  predicted_next
0       0.4537        0.245978
1       0.6088        0.285932
2       0.6713        0.446064
3       0.6509        0.613396
4       0.3922        0.629307
5       0.2672        0.483490
6       0.2188        0.508382
7       0.1810        0.195090
8       0.1606        0.293324
9       0.1562        0.171839


In [ ]:
### 3 benchmarking models


# 1. IMPORTS

import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor

# 3. DEFINE MODELS

models = {
    "LightGBM": LGBMRegressor(
        n_estimators=700,
        learning_rate=0.04,
        max_depth=6,
        num_leaves=45,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=0.2,
        random_state=42
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),

    "Ridge": Ridge(
        alpha=1.0
    )
}



# 4. EVALUATION FUNCTION

def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test):

    model.fit(X_train, y_train)

    pred_train = model.predict(X_train)
    pred_val   = model.predict(X_val)
    pred_test  = model.predict(X_test)

    results = {
        "Train_MAE": mean_absolute_error(y_train, pred_train),
        "Val_MAE":   mean_absolute_error(y_val, pred_val),
        "Test_MAE":  mean_absolute_error(y_test, pred_test),

        "Train_RMSE": np.sqrt(mean_squared_error(y_train, pred_train)),
        "Val_RMSE":   np.sqrt(mean_squared_error(y_val, pred_val)),
        "Test_RMSE":  np.sqrt(mean_squared_error(y_test, pred_test)),
    }

    return results



# 5. RUN BENCHMARK

benchmark_results = []

for name, model in models.items():
    print(f"\nRunning {name}...")

    res = evaluate_model(
        model,
        X_train, y_train,
        X_val, y_val,
        X_test, y_test
    )

    res["Model"] = name
    benchmark_results.append(res)



# 6. RESULTS TABLE

results_df = pd.DataFrame(benchmark_results)

# reorder columns
results_df = results_df[[
    "Model",
    "Train_MAE", "Val_MAE", "Test_MAE",
    "Train_RMSE", "Val_RMSE", "Test_RMSE"
]]

print("\n==============================")
print(" MODEL BENCHMARK RESULTS")
print("==============================")
print(results_df)



# 7. PSEUDO ACCURACY

results_df["Pseudo_Accuracy"] = (1 - results_df["Test_MAE"]) * 100

print("\n=== WITH PSEUDO ACCURACY ===")
print(results_df[["Model", "Test_MAE", "Pseudo_Accuracy"]])



# 8. SORTED VIEW OF BEST MODEL FIRST

results_sorted = results_df.sort_values("Test_MAE")

print("\n=== BEST MODEL RANKING ===")
print(results_sorted[["Model", "Test_MAE", "Pseudo_Accuracy"]])


Running LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001568 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1451
[LightGBM] [Info] Number of data points in the train set: 16984, number of used features: 18
[LightGBM] [Info] Start training from score 0.783162
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posi

In [ ]:

# 0. IMPORTS

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from datetime import timedelta



# 1. NAME FIX

name_fix_map = {
    "Victoria St": "Victoria Street"
}



# 2. TOTAL SPACES (CIVIC INCLUDED)

total_spaces_map = {
    "Civic": 930,
    "Ronwood": 678,
    "Toka Puia": 420,
    "Victoria Street": 895
}



# 3. FETCH LIVE DATA

url = "https://at.govt.nz/umbraco/surface/parkingavailabilitysurface/ParkingAvailabilityResult"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")



# 4. PARSE HTML

rows = soup.find_all("div", class_="divTableRow")

data = []

for row in rows:
    cells = row.find_all("div", class_="divTableCell")

    if len(cells) < 3:
        continue

    location = cells[0].get_text(strip=True)
    ptype = cells[1].get_text(strip=True)
    available = cells[2].get_text(strip=True)

    if location == "Car park":
        continue

    if ptype != "Short-term":
        continue

    try:
        available = int(available)
    except:
        continue

    # FIX NAME
    location = name_fix_map.get(location, location)

    data.append({
        "location_id": location,
        "available_spaces": available
    })

live_df = pd.DataFrame(data)

print("\n===== LIVE DATA =====")
print(live_df)



# 5. FILTER TARGET LOCATIONS (CIVIC VERSION)

live_df = live_df[live_df["location_id"].isin(total_spaces_map.keys())]



# 6. ADD TOTAL SPACES

live_df["total_spaces"] = live_df["location_id"].map(total_spaces_map)



# 7. CURRENT OCCUPANCY

live_df["occupancy_now"] = 1 - (live_df["available_spaces"] / live_df["total_spaces"])
live_df["occupancy_now"] = live_df["occupancy_now"].clip(0, 1)



# 8. HISTORICAL DATA

hist_df = akl_final.copy()
hist_df = hist_df.sort_values(["location_id", "timestamp_hour"])



# 9. BUILD LAG FEATURES

rows = []

for _, row in live_df.iterrows():
    loc = row["location_id"]

    hist_loc = hist_df[hist_df["location_id"] == loc]

    if len(hist_loc) < 2:
        print(f"⚠️ Skipping {loc} (no sufficient history)")
        continue

    last2 = hist_loc.tail(2)["occupancy_rate"].values

    lag_1 = row["occupancy_now"]
    lag_2 = last2[-1]
    lag_3 = last2[-2]

    rolling_mean_3 = (lag_1 + lag_2 + lag_3) / 3

    rows.append({
        "location_id": loc,
        "lag_1": lag_1,
        "lag_2": lag_2,
        "lag_3": lag_3,
        "rolling_mean_3": rolling_mean_3,
        "occupancy_now": lag_1
    })

live_features = pd.DataFrame(rows)

if live_features.empty:
    raise ValueError("❌ No valid locations with history found")



# 10. TIME FEATURES

now = pd.Timestamp.now().floor("h")
future_time = now + timedelta(hours=1)

live_features["hour"] = future_time.hour
live_features["day_of_week"] = future_time.dayofweek
live_features["month"] = future_time.month
live_features["is_weekend"] = 1 if future_time.dayofweek >= 5 else 0

live_features["hour_sin"] = np.sin(2*np.pi*live_features["hour"]/24)
live_features["hour_cos"] = np.cos(2*np.pi*live_features["hour"]/24)

live_features["hour_weekend"] = live_features["hour"] * live_features["is_weekend"]



# 11. WEATHER

latest_weather = hist_df.sort_values("timestamp_hour").iloc[-1]

rain_map = {"no_rain": 0, "light": 1, "moderate": 2}

live_features["temperature"] = latest_weather["temperature"]
live_features["precipitation"] = latest_weather["precipitation"]
live_features["is_rain"] = latest_weather["is_rain"]
live_features["rain_level"] = rain_map.get(latest_weather["rain_level"], 0)

live_features["rain_hour"] = live_features["hour"] * live_features["is_rain"]
live_features["temp_hour"] = live_features["hour"] * live_features["temperature"]



# 12. LOCATION ENCODING

location_map = dict(zip(
    akl_final_ready["location_id"],
    akl_final_ready["location_encoded"]
))

live_features["location_encoded"] = live_features["location_id"].map(location_map).fillna(-1)



# 13. FEATURES

features = [
    "location_encoded","hour","day_of_week","month","is_weekend",
    "temperature","precipitation","is_rain","rain_level",
    "hour_sin","hour_cos","hour_weekend","rain_hour","temp_hour",
    "lag_1","lag_2","lag_3","rolling_mean_3"
]

X_live = live_features[features]



# 14. PREDICT

pred = model.predict(X_live)

live_features["predicted_next_hour"] = pred



# 15. VEHICLE COUNT

live_features["predicted_cars"] = (
    live_features["predicted_next_hour"] *
    live_features["location_id"].map(total_spaces_map)
).round(0)



# 16. OUTPUT

print("\n==============================")
print(" LIVE PREDICTION (CIVIC VERSION)")
print("==============================")

print(live_features[[
    "location_id",
    "occupancy_now",
    "predicted_next_hour",
    "predicted_cars"
]])


===== LIVE DATA =====
       location_id  available_spaces
0    Albert Street                 0
1            Civic               822
2         Downtown                 0
3          Ronwood               528
4        Toka Puia               395
5  Victoria Street               595

 LIVE PREDICTION (CIVIC VERSION)
       location_id  occupancy_now  predicted_next_hour  predicted_cars
0            Civic       0.116129             0.556284           517.0
1          Ronwood       0.221239             0.532230           361.0
2        Toka Puia       0.059524             0.418161           176.0
3  Victoria Street       0.335196             0.729735           653.0
